In [1]:
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from osgeo import gdal
from rasterio.warp import reproject, Resampling
gdal.UseExceptions()
import importlib
import sys
sys.path.append('../scripts')
import model_comparison_functions
importlib.reload(model_comparison_functions)
from model_comparison_functions import process_all_dates, get_raw_data

In [7]:
# task1 = process_all_dates("C:/Users/13038/Desktop/SIRO/Model_Outputs/dates/dates", 1)
# task2 = process_all_dates("C:/Users/13038/Desktop/SIRO/Model_Outputs/dates/dates", 2)
task1 = process_all_dates("/Users/rdcrlrka/Research/SIRO/MCSModeling/dates", 1)
task2 = process_all_dates("/Users/rdcrlrka/Research/SIRO/MCSModeling/dates", 2)

task1

{'20240315': {'SNODAS-basin': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20240315/modeled/SNODAS/SNODAS_20240315_basin_clip.tif',
  'SNODAS-MCS': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20240315/modeled/SNODAS/SNODAS_20240315_MCS_clip.tif'},
 '20250501': {'SNODAS-basin': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20250501/modeled/SNODAS/SNODAS_20250501_basin_clip.tif',
  'SNODAS-MCS': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20250501/modeled/SNODAS/SNODAS_20250501_MCS_clip.tif'},
 '20240418': {'SNODAS-basin': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20240418/modeled/SNODAS/SNODAS_20240418_basin_clip.tif',
  'SNODAS-MCS': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20240418/modeled/SNODAS/SNODAS_20240418_MCS_clip.tif'},
 '20250404': {'SNODAS-basin': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20250404/modeled/SNODAS/SNODAS_20250404_basin_clip.tif',
  'SNODAS-MCS': '/Users/rdcrlrka/Research/SIRO/MCSModeling/dates/20250404/modeled/SNODAS/SNOD

## LiDAR Domain Plots

In [6]:
dfs = []

for key in task1.keys():
    task1_clip = task1[key]['lidar_clip']
    task2_clip = task2[key]['lidar_clip']


    MCS = {"Task 1":task1_clip, "Task 2":task2_clip, }

    for task, raster_list in MCS.items():
        for name, model_data in raster_list.items():
                with rasterio.open(model_data) as src:
                    data = src.read(1)
                    mask = (data == -9999)
                    data_masked = np.ma.array(data, mask=mask)
                    flattened = data_masked.compressed()
                    flattened = flattened[flattened < 5]

                    df = pd.DataFrame({
                    "Task":task,
                    "Date":key,
                    "Model": name,        # this column will store model names
                    "Snow Depth": flattened,
                    })
                    dfs.append(df)


lidar_domain = pd.concat(dfs, ignore_index=True)

KeyError: 'lidar_clip'

In [ ]:
Task1_lidar = lidar_domain[lidar_domain['Task'] =='Task 1']
Task2_lidar = lidar_domain[lidar_domain['Task'] =='Task 2']

In [ ]:

fig, ax = plt.subplots(figsize=(15, 8))

order = ['LiDAR', 'iSnobal', 'SnowModel', 'LiDAR-2000', 'HMS-EB', 'HMS-TI']

sns.boxplot(
    data=Task1_lidar,
    x = "Date",
    y= "Snow Depth",
    hue = "Model",
    hue_order = order,
    gap = 0.1)


ax.set_xlabel('Date', fontsize = 22)
ax.set_ylabel('Snow Depth (m)', fontsize = 22)
ax.tick_params(axis='both', which='major', labelsize=20)
ax.set_xticklabels(['4-23', '3-24', '4-24', '4-25', '5-25'])
ax.legend(prop={'size': 20}, ncol = 2) # Sets exact font size



plt.tight_layout()
plt.savefig("C:/Users/13038/Desktop/SIRO/docs/docs/figs/lidar_distributions.png",
            dpi = 300
            )
plt.show()

In [ ]:
df = lidar_domain[
    (lidar_domain['Model'] == 'SnowModel') |
    (lidar_domain['Model'] == 'iSnobal')
]

df['Model_Task'] =df['Model'] + ' - ' + df['Task']

palette = {
    'iSnobal - Task 1': 'darkorange',
    'iSnobal - Task 2': 'moccasin',   # light orange

    'SnowModel - Task 1': 'green',
    'SnowModel - Task 2': 'honeydew'  # light green
}

order = ['iSnobal - Task 1', 'iSnobal - Task 2', 'SnowModel - Task 1', 'SnowModel - Task 2']


In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))


sns.boxplot(
    data=df,
    x = "Date",
    y= "Snow Depth",
    hue = "Model_Task",
    palette=palette,
    hue_order = order,
    gap = 0.1)


ax.set_xlabel('Date', fontsize = 22)
ax.set_ylabel('Snow Depth (m)', fontsize = 22)
ax.tick_params(axis='both', which='major', labelsize=20)
ax.set_xticklabels(['4-23', '3-24', '4-24', '4-25', '5-25'])
ax.legend(prop={'size': 20}, ncol = 2) # Sets exact font size
ax.set_title(" ", fontsize=24)



plt.tight_layout()
#plt.savefig("C:/Users/RDCRLSMC/Desktop/SIRO/docs/figs/lidar_distributions_tasks.png", dpi = 300)
plt.show()

## Basin Plots

In [10]:
dfs = []

for key in task1.keys():
    task1_clip = task1[key]['basin_clip']
    task2_clip = task2[key]['basin_clip']

    MCS = {"Task 1":task1_clip, "Task 2":task2_clip}


    for task, raster_list in MCS.items():
        for name, model_data in raster_list.items():
                with rasterio.open(model_data) as src:

                    if "HMS" in name:
                        target_res = 100  # meters

                        xres = src.transform.a
                        scale = xres / target_res

                        new_height = int(src.height * scale)
                        new_width = int(src.width * scale)



                        data = src.read(
                            1,
                            out_shape=(new_height, new_width),
                            resampling=Resampling.nearest
                        )

                        bounds = src.bounds

                        new_xres = (bounds.right - bounds.left) / new_width
                        new_yres = (bounds.top - bounds.bottom) / new_height

                        mask = (data == -9999)
                        data_masked = np.ma.array(data, mask=mask)
                        flattened = data_masked.compressed()
                        flattened = flattened[flattened < 5]

                    else:
                        data = src.read(1)
                        mask = (data == -9999)
                        data_masked = np.ma.array(data, mask=mask)
                        flattened = data_masked.compressed()
                        flattened = flattened[flattened < 5]

                    df = pd.DataFrame({
                    "Task":task,
                    "Date":key,
                    "Model": name,        # this column will store model names
                    "Snow Depth": flattened,
                    "measured": name == "lidar"# this column stores raster values
                    })
                    dfs.append(df)


basin_domain = pd.concat(dfs, ignore_index=True)

C:\Users\13038\AppData\Local\Temp\ipykernel_6784\3489409438.py:25: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = src.read(
C:\Users\13038\AppData\Local\Temp\ipykernel_6784\3489409438.py:42: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = src.read(1)


In [ ]:
Task1_basin = basin_domain[basin_domain['Task'] =='Task 1']
Task2_basin = basin_domain[basin_domain['Task'] =='Task 2']

In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

fig, ax = plt.subplots(figsize=(15, 8))
order = ['','HMS-EB', 'HMS-TI', 'iSnobal', 'SnowModel']

sns.boxplot(
    data=Task1_basin,
    x = "Date",
    y= "Snow Depth",
    hue = "Model",
   hue_order = order,
    gap = 0.1)


ax.set_xlabel('Date', fontsize = 22)
ax.set_ylabel('Snow Depth (m)', fontsize = 22)
ax.tick_params(axis='both', which='major', labelsize=20)
ax.set_xticklabels(['4-23', '3-24', '4-24', '4-25', '5-25'])
ax.legend(prop={'size': 20}, ncol = 2) # Sets exact font size
ax.set_title("Mores Creek Watershed Modeled Snow Depth ", fontsize=24)



plt.tight_layout()
#plt.savefig("C:/Users/RDCRLSMC/Desktop/SIRO/docs/figs/basin_distributions.png", dpi = 300)
plt.show()

## By elevation

In [ ]:
dem_fp = "C:/Users/RDCRLSMC/Desktop/SIRO/data/shapes/merged_32611_clip.tif"


dfs = []
with rasterio.open(dem_fp) as src:
    dem = src.read(1)
    dem = dem[dem != 0]
    low_thr, high_thr = np.percentile(dem, [33.33, 66.66])
    print(low_thr, high_thr)
    print(dem.min(), dem.max())

for key in task1.keys():
    task1_clip = task1[key]['basin_clip']
    task2_clip = task2[key]['basin_clip']

    MCS = {"Task 1":task1_clip, "Task 2":task2_clip}

    for task, raster_list in MCS.items():
        for name, model_data in raster_list.items():
                with rasterio.open(model_data) as src:
                    ref_shape = (src.height, src.width)
                    ref_transform = src.transform
                    data = src.read(1)
                    # mask = (data == -9999)
                    # data_masked = np.ma.array(data, mask=mask)
                    # flattened = data_masked.compressed()
                    # flattened = flattened[flattened < 5]

                elevation_aligned = np.empty(ref_shape, dtype=np.float32)

                with rasterio.open(dem_fp) as dem:
                    reproject(
                        source=dem.read(1),
                        destination=elevation_aligned,
                        src_transform=dem.transform,
                        src_crs=dem.crs,
                        dst_transform=ref_transform,
                        dst_crs="EPSG:32611",
                        resampling=Resampling.nearest
                    )

                    invalid_snow = (data == -9999) | (data >= 5)
                    invalid_elev = (elevation_aligned == 0)
                    master_mask = invalid_snow | invalid_elev

                    data_masked = np.ma.array(data, mask=master_mask)
                    elevation_masked = np.ma.array(elevation_aligned, mask=master_mask)

                    # Compress them (they are guaranteed to be the same length now)
                    flattened = data_masked.compressed()
                    elevation_flat = elevation_masked.compressed()

                    # elevation_masked = np.ma.array(elevation_aligned, mask=mask)
                    # elevation_masked.mask |= (elevation_masked == 0)
                    # elevation_flat = elevation_masked.compressed()
                    elev_class = np.where(
                                elevation_flat < low_thr, "low",
                                np.where(elevation_flat < high_thr, "middle", "high")
                            )
                    df = pd.DataFrame({
                    "Task":task,
                    "Date":key,
                    "Model": name,        # this column will store model names
                    "Snow Depth": flattened,
                    "Elevation": elevation_flat,
                    "Elevation Class": elev_class
                    })
                    dfs.append(df)


snow_depth = pd.concat(dfs, ignore_index=True)

In [ ]:
Task1_basin = snow_depth[snow_depth['Task'] =='Task 1']
Task2_basin = snow_depth[snow_depth['Task'] =='Task 2']

In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

fig, ax = plt.subplots(figsize=(15, 8))

sns.boxplot(
    data=Task2_basin[Task2_basin['Date'] =='20230405'],
    x = "Elevation Class",
    y= "Snow Depth",
    order = ['low', 'middle', 'high'],
    hue = "Model",
    gap = 0.1)


ax.set_xlabel('Date', fontsize = 18)
ax.set_ylabel('Snow Depth (m)', fontsize = 18)
ax.tick_params(axis='both', which='major', labelsize=16)
ax.legend(prop={'size': 16}) # Sets exact font size
ax.set_title("Modled Snow Depth by Elevation Class in Mores Creek Basin, April 5 2023 ", fontsize=20)



plt.tight_layout()
plt.show()